In [52]:
import pandas as pd

In [53]:
# -----------------------------------------
# STEP 1: Loading PaySim dataset
# -----------------------------------------

df = pd.read_csv("../../data/raw//PS_20174392719_1491204439457_log.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (6362620, 11)


In [54]:
# -----------------------------------------
# STEP 2: Balance Consistency Investigation
# -----------------------------------------

# Origin account balance difference
df["origin_balance_diff"] = ( 
    df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"]
)

# Destination account balance difference
df["destination_balance_diff"] = (
    df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]
)

# Summary statistics
print("\nOrigin Balance Difference:")
print(df["origin_balance_diff"].describe())

print("\nDestination Balance Difference:")
print(df["destination_balance_diff"].describe())



Origin Balance Difference:
count    6.362620e+06
mean    -2.010925e+05
std      6.066505e+05
min     -9.244552e+07
25%     -2.496411e+05
50%     -6.867726e+04
75%     -2.954230e+03
max      1.000000e-02
Name: origin_balance_diff, dtype: float64

Destination Balance Difference:
count    6.362620e+06
mean     5.556717e+04
std      4.415288e+05
min     -7.588573e+07
25%      0.000000e+00
50%      3.500490e+03
75%      2.935305e+04
max      1.319123e+07
Name: destination_balance_diff, dtype: float64


In [55]:
# -----------------------------------------
# STEP 3: Balance Consistency by Transaction Type
# -----------------------------------------

balance_check = (
    df.groupby("type").agg(
        transactions=("type","size"),
        origin_mean_diff=("origin_balance_diff","mean"),
        destination_mean_diff=("destination_balance_diff","mean"),
    ).round(2)
)

print("\nBalance Consistency by Transaction Type:")
print(balance_check)



Balance Consistency by Transaction Type:
          transactions  origin_mean_diff  destination_mean_diff
type                                                           
CASH_IN        1399284        -337835.45              289733.65
CASH_OUT       2237500        -147724.35              -17294.22
DEBIT            41432          -1997.98              -14384.03
PAYMENT        2151495          -6678.67               13057.60
TRANSFER        532909        -866493.31              -76314.10


In [56]:
# -----------------------------------------
# STEP 4: Zero Balance Investigation
# -----------------------------------------

balance_columns = [
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

zero_balance_summary = (
    df[balance_columns].eq(0).mean().mul(100).round(2)
)

print("\nPercentage of Zero Values in Balance Columns:")
print(zero_balance_summary)



Percentage of Zero Values in Balance Columns:
oldbalanceOrg     33.04
newbalanceOrig    56.73
oldbalanceDest    42.50
newbalanceDest    38.34
dtype: float64


In [57]:
# -----------------------------------------
# STEP 5: Zero Balances by Transaction Type
# -----------------------------------------

zero_by_type = (
    df.groupby("type")[balance_columns]
      .apply(lambda x: (x == 0).mean() * 100)
      .round(2)
)

print("\nPercentage of Zero Balances by Transaction Type:")
print(zero_by_type)



Percentage of Zero Balances by Transaction Type:
          oldbalanceOrg  newbalanceOrig  oldbalanceDest  newbalanceDest
type                                                                   
CASH_IN            0.96            0.00           11.69           19.36
CASH_OUT          45.85           88.72           14.49            0.51
DEBIT             14.86           28.45            0.00            1.13
PAYMENT           35.99           51.18          100.00          100.00
TRANSFER          53.06           95.97           12.22            0.97


In [58]:
# -----------------------------------------
# STEP 6: Fraud Distribution by Transaction Type
# -----------------------------------------

fraud_by_type = (
    df.groupby("type")
      .agg(
          transactions=("isFraud", "size"),
          fraud_transactions=("isFraud", "sum"),
          fraud_rate=("isFraud", "mean")
      )
)

fraud_by_type["fraud_rate"] = (
    fraud_by_type["fraud_rate"] * 100
).round(4)

print("\nFraud Distribution by Transaction Type:")
print(fraud_by_type)


Fraud Distribution by Transaction Type:
          transactions  fraud_transactions  fraud_rate
type                                                  
CASH_IN        1399284                   0      0.0000
CASH_OUT       2237500                4116      0.1840
DEBIT            41432                   0      0.0000
PAYMENT        2151495                   0      0.0000
TRANSFER        532909                4097      0.7688


In [59]:
# -----------------------------------------
# STEP 7: isFlaggedFraud vs isFraud
# -----------------------------------------

flagged_summary = (
    df.groupby("isFlaggedFraud")["isFraud"]
      .agg(
          transactions="count",
          actual_fraud="sum"
      )
)

flagged_summary["fraud_rate"] = (
    flagged_summary["actual_fraud"]
    / flagged_summary["transactions"]
    * 100
).round(4)

print("\nExisting Fraud Flag vs Actual Fraud:")
print(flagged_summary)


Existing Fraud Flag vs Actual Fraud:
                transactions  actual_fraud  fraud_rate
isFlaggedFraud                                        
0                    6362604          8197      0.1288
1                         16            16    100.0000


In [60]:
# -----------------------------------------
# STEP 8: Duplicate Transaction Investigation
# -----------------------------------------

duplicate_count = df.duplicated().sum()

print("\nExact Duplicate Transactions:")
print("Duplicate rows:", duplicate_count)

# Check duplicates excluding engineered investigation columns
original_columns = [
    "step",
    "type",
    "amount",
    "nameOrig",
    "oldbalanceOrg",
    "newbalanceOrig",
    "nameDest",
    "oldbalanceDest",
    "newbalanceDest",
    "isFraud",
    "isFlaggedFraud"
]

duplicate_original = df.duplicated(
    subset=original_columns
).sum()

print("Duplicate rows based on original columns:", duplicate_original)


Exact Duplicate Transactions:
Duplicate rows: 0
Duplicate rows based on original columns: 0


In [61]:
# -----------------------------------------
# STEP 9: Negative Financial Values
# -----------------------------------------

financial_columns = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

negative_values = (
    df[financial_columns]
    .lt(0)
    .sum()
)

print("\nNegative Values in Financial Columns:")
print(negative_values)


Negative Values in Financial Columns:
amount            0
oldbalanceOrg     0
newbalanceOrig    0
oldbalanceDest    0
newbalanceDest    0
dtype: int64


In [62]:
# -----------------------------------------
# STEP 10: Zero-Amount Transaction Investigation
# -----------------------------------------

zero_amount_summary = (
    df.groupby("type")
      .agg(
          transactions=("amount", "size"),
          zero_amount_transactions=("amount", lambda x: (x == 0).sum()),
          fraud_transactions=("isFraud", "sum")
      )
)

zero_amount_summary["zero_amount_rate"] = (
    zero_amount_summary["zero_amount_transactions"]
    / zero_amount_summary["transactions"]
    * 100
).round(4)

print("\nZero-Amount Transactions by Type:")
print(zero_amount_summary)


Zero-Amount Transactions by Type:
          transactions  zero_amount_transactions  fraud_transactions  \
type                                                                   
CASH_IN        1399284                         0                   0   
CASH_OUT       2237500                        16                4116   
DEBIT            41432                         0                   0   
PAYMENT        2151495                         0                   0   
TRANSFER        532909                         0                4097   

          zero_amount_rate  
type                        
CASH_IN             0.0000  
CASH_OUT            0.0007  
DEBIT               0.0000  
PAYMENT             0.0000  
TRANSFER            0.0000  


In [63]:
# -----------------------------------------
# STEP 11: Transaction Amount Distribution
# -----------------------------------------

amount_summary = df["amount"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
)

print("\nTransaction Amount Distribution:")
print(amount_summary)


Transaction Amount Distribution:
count    6.362620e+06
mean     1.798619e+05
std      6.038582e+05
min      0.000000e+00
25%      1.338957e+04
50%      7.487194e+04
75%      2.087215e+05
90%      3.654233e+05
95%      5.186342e+05
99%      1.615979e+06
99.9%    8.956798e+06
max      9.244552e+07
Name: amount, dtype: float64


In [64]:
# -----------------------------------------
# STEP 12: Origin Account Transaction Behavior
# -----------------------------------------

origin_transaction_count = (
    df.groupby("nameOrig")
      .size()
)

print("\nOrigin Account Transaction Count:")
print(origin_transaction_count.describe())


Origin Account Transaction Count:
count    6.353307e+06
mean     1.001466e+00
std      3.832002e-02
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      3.000000e+00
dtype: float64


In [65]:
# -----------------------------------------
# STEP 13: Origin Account Behavior vs Fraud
# -----------------------------------------

df["origin_transaction_count"] = (
    df.groupby("nameOrig")["nameOrig"]
      .transform("size")
)

origin_behavior_by_fraud = (
    df.groupby("isFraud")["origin_transaction_count"]
      .agg(
          mean="mean",
          median="median",
          max="max"
      )
      .round(4)
)

print("\nOrigin Account Behavior by Fraud Status:")
print(origin_behavior_by_fraud)


Origin Account Behavior by Fraud Status:
           mean  median  max
isFraud                     
0        1.0029     1.0    3
1        1.0034     1.0    2


In [66]:
# -----------------------------------------
# STEP 14: Destination Account Behavior
# -----------------------------------------

destination_transaction_count = (
    df.groupby("nameDest")
      .size()
)

print("\nDestination Account Transaction Count:")
print(destination_transaction_count.describe())


Destination Account Transaction Count:
count    2.722362e+06
mean     2.337169e+00
std      4.549264e+00
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.130000e+02
dtype: float64


In [67]:
# -----------------------------------------
# STEP 15: Destination Account Behavior vs Fraud
# -----------------------------------------

df["destination_transaction_count"] = (
    df.groupby("nameDest")["nameDest"]
      .transform("size")
)

destination_behavior_by_fraud = (
    df.groupby("isFraud")["destination_transaction_count"]
      .agg(
          mean="mean",
          median="median",
          max="max"
      )
      .round(4)
)

print("\nDestination Account Behavior by Fraud Status:")
print(destination_behavior_by_fraud)


Destination Account Behavior by Fraud Status:
            mean  median  max
isFraud                      
0        11.1962     7.0  113
1         8.0953     4.0   89


In [68]:
# -----------------------------------------
# STEP 16: Transaction Amount vs Fraud
# -----------------------------------------

amount_by_fraud = (
    df.groupby("isFraud")["amount"]
      .describe(
          percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
      )
      .round(2)
)

print("\nTransaction Amount by Fraud Status:")
print(amount_by_fraud)


Transaction Amount by Fraud Status:
             count        mean         std   min        25%        50%  \
isFraud                                                                  
0        6354407.0   178197.04   596236.98  0.01   13368.40   74684.72   
1           8213.0  1467967.30  2404252.95  0.00  127091.33  441423.44   

                75%         90%         95%          99%          max  
isFraud                                                                
0         208364.76   364373.44   515610.42   1586064.17  92445516.64  
1        1517771.48  4521723.51  8006429.04  10000000.00  10000000.00  


In [69]:
# -----------------------------------------
# STEP 17: Transaction Amount Relative to Origin Balance
# -----------------------------------------

df["amount_to_origin_balance"] = (
    df["amount"] /
    df["oldbalanceOrg"].replace(0, pd.NA)
)

amount_ratio_by_fraud = (
    df.groupby("isFraud")["amount_to_origin_balance"]
      .describe(
          percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
      )
      .round(4)
)

print("\nAmount-to-Origin-Balance Ratio by Fraud Status:")
print(amount_ratio_by_fraud)


Amount-to-Origin-Balance Ratio by Fraud Status:
             count     unique     top    freq
isFraud                                      
0        4251999.0  4251847.0  31.069     2.0
1           8172.0      155.0   1.000  8018.0


In [70]:
# -----------------------------------------
# STEP 18: Full Origin Balance Utilization
# -----------------------------------------

df["amount_equals_origin_balance"] = (
    df["amount"] == df["oldbalanceOrg"]
)

ratio_flag_summary = (
    df.groupby("amount_equals_origin_balance")["isFraud"]
      .agg(
          transactions="count",
          fraud_transactions="sum"
      )
)

ratio_flag_summary["fraud_rate"] = (
    ratio_flag_summary["fraud_transactions"]
    / ratio_flag_summary["transactions"]
    * 100
).round(4)

print("\nAmount Equals Origin Balance vs Fraud:")
print(ratio_flag_summary)


Amount Equals Origin Balance vs Fraud:
                              transactions  fraud_transactions  fraud_rate
amount_equals_origin_balance                                              
False                              6354586                 179      0.0028
True                                  8034                8034    100.0000


In [71]:
# -----------------------------------------
# STEP 19: Full Balance Utilization by Type
# -----------------------------------------

balance_utilization_by_type = (
    df.groupby("type")
      .agg(
          transactions=("isFraud", "size"),
          full_balance_transactions=("amount_equals_origin_balance", "sum"),
          fraud_transactions=("isFraud", "sum")
      )
)

balance_utilization_by_type["full_balance_rate"] = (
    balance_utilization_by_type["full_balance_transactions"]
    / balance_utilization_by_type["transactions"]
    * 100
).round(4)

print("\nFull Origin Balance Utilization by Transaction Type:")
print(balance_utilization_by_type)


Full Origin Balance Utilization by Transaction Type:
          transactions  full_balance_transactions  fraud_transactions  \
type                                                                    
CASH_IN        1399284                          0                   0   
CASH_OUT       2237500                       4091                4116   
DEBIT            41432                          0                   0   
PAYMENT        2151495                          0                   0   
TRANSFER        532909                       3943                4097   

          full_balance_rate  
type                         
CASH_IN              0.0000  
CASH_OUT             0.1828  
DEBIT                0.0000  
PAYMENT              0.0000  
TRANSFER             0.7399  


In [72]:
# -----------------------------------------
# STEP 20: Legitimate Full-Balance Transactions
# -----------------------------------------

legit_full_balance = df[
    (df["amount_equals_origin_balance"] == True) &
    (df["isFraud"] == 0)
]

legit_full_balance_summary = (
    legit_full_balance.groupby("type")
      .size()
      .to_frame("legitimate_full_balance_transactions")
)

In [73]:
# -----------------------------------------
# STEP 21: Remaining Fraud Transactions (179)
# -----------------------------------------

remaining_fraud = df[
    (df["isFraud"] == 1) &
    (df["amount_equals_origin_balance"] == False)
]

remaining_fraud_summary = (
    remaining_fraud.groupby("type")
    .agg(
        transactions=("isFraud", "size"),
        mean_amount=("amount", "mean"),
        median_amount=("amount", "median"),
        mean_origin_balance=("oldbalanceOrg", "mean"),
        mean_destination_balance=("oldbalanceDest", "mean")
    )
    .round(2)
)

print("\nRemaining Fraud Transactions:")
print(remaining_fraud_summary)

print("\nTotal Remaining Fraud Transactions:")
print(len(remaining_fraud))


Remaining Fraud Transactions:
          transactions  mean_amount  median_amount  mean_origin_balance  \
type                                                                      
CASH_OUT            25    220121.42      181728.11             17031.66   
TRANSFER           154   9565121.77    10000000.00          19288380.61   

          mean_destination_balance  
type                                
CASH_OUT                 580666.89  
TRANSFER                  28556.18  

Total Remaining Fraud Transactions:
179


In [75]:
# -----------------------------------------
# STEP 22: Remaining Fraud Amount Distribution
# -----------------------------------------

remaining_fraud_amount = (
    remaining_fraud["amount"]
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
    .round(2)
)

print("\nRemaining Fraud Amount Distribution:")
print(remaining_fraud_amount)


Remaining Fraud Amount Distribution:
count         179.00
mean      8259954.12
std       3704432.87
min         23292.30
25%      10000000.00
50%      10000000.00
75%      10000000.00
90%      10000000.00
95%      10000000.00
99%      10000000.00
max      10000000.00
Name: amount, dtype: float64


In [76]:
# -----------------------------------------
# STEP 23: Exact 10M Fraud Investigation
# -----------------------------------------

ten_million_fraud = remaining_fraud[
    remaining_fraud["amount"] == 10_000_000
]

print("\nRemaining Fraud Transactions at Exactly 10M:")
print("Count:", len(ten_million_fraud))

print("\nPercentage of Remaining Fraud at Exactly 10M:")
print(
    round(
        len(ten_million_fraud) / len(remaining_fraud) * 100,
        2
    ),
    "%"
)


Remaining Fraud Transactions at Exactly 10M:
Count: 145

Percentage of Remaining Fraud at Exactly 10M:
81.01 %


#### Remaining 34 fraud , neither amount == oldbalanceOrg nor amount == 10,000,000

In [77]:
# -----------------------------------------
# STEP 24: The Remaining 34 Fraud Cases
# -----------------------------------------

remaining_34_fraud = df[
    (df["isFraud"] == 1) &
    (df["amount_equals_origin_balance"] == False) &
    (df["amount"] != 10_000_000)
]

remaining_34_summary = (
    remaining_34_fraud.groupby("type")
    .agg(
        transactions=("isFraud", "size"),
        mean_amount=("amount", "mean"),
        median_amount=("amount", "median"),
        min_amount=("amount", "min"),
        max_amount=("amount", "max"),
        mean_origin_balance=("oldbalanceOrg", "mean"),
        mean_destination_balance=("oldbalanceDest", "mean")
    )
    .round(2)
)

print("\nRemaining 34 Fraud Transactions:")
print(remaining_34_summary)

print("\nTotal Remaining Fraud Transactions:")
print(len(remaining_34_fraud))


Remaining 34 Fraud Transactions:
          transactions  mean_amount  median_amount  min_amount  max_amount  \
type                                                                         
CASH_OUT            25    220121.42      181728.11    23292.30   577418.98   
TRANSFER             9   2558750.27     1078013.76   123194.95  9585040.37   

          mean_origin_balance  mean_destination_balance  
type                                                     
CASH_OUT             17031.66                 580666.89  
TRANSFER           5424130.87                 488627.95  

Total Remaining Fraud Transactions:
34


In [78]:
# -----------------------------------------
# STEP 25: Inspect Individual Remaining 34 Fraud Cases
# -----------------------------------------

print("\nDetailed Remaining 34 Fraud Transactions:")

print(
    remaining_34_fraud[
        [
            "step",
            "type",
            "amount",
            "nameOrig",
            "oldbalanceOrg",
            "newbalanceOrig",
            "nameDest",
            "oldbalanceDest",
            "newbalanceDest"
        ]
    ]
    .sort_values("amount", ascending=False)
    .to_string(index=False)
)


Detailed Remaining 34 Fraud Transactions:
 step     type     amount    nameOrig  oldbalanceOrg  newbalanceOrig    nameDest  oldbalanceDest  newbalanceDest
  425 TRANSFER 9585040.37  C452586515    19585040.37     19585040.37 C1109166882            0.00            0.00
  730 TRANSFER 7316255.05 C1869569059    17316255.05     17316255.05 C1861208726            0.00            0.00
   11 TRANSFER 1933920.80 C1706582969           0.00            0.00  C461905695      1283762.85      3217683.65
   43 TRANSFER 1395850.55 C1296215617           0.00            0.00 C1429415136       260806.21      1656656.77
    8 TRANSFER 1078013.76 C1026280121           0.00            0.00  C277510102            0.00       970749.68
    9 TRANSFER  994453.20 C1121789613     1437370.87       442917.67  C254839817       194812.76       665743.67
   38 CASH_OUT  577418.98 C1907944035           0.00            0.00  C541373010            0.00       577418.98
   18 CASH_OUT  508782.20  C576782065           0.00 

PaySim contains substantial structural balance inconsistencies and many legitimate zero-balance states. Fraud is highly concentrated in CASH_OUT and TRANSFER transactions. The dataset exhibits strong fraud patterns involving complete origin-balance depletion and high-value transactions, while a small number of fraud cases exhibit additional balance and behavioral anomalies.